# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, referencing data entities solely by their Croissant `@id` fields. Follow along for practical steps in analysis and visualization.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. We'll work with metadata and data records using Croissant `@id` fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Citation: {dataset.metadata.cite_as}")
print(f"License: {dataset.metadata.license}")
print(f"Published: {dataset.metadata.date_published}")

## 2. Data Overview
Let's review available record sets in the dataset and their field `@id`s.

In [ ]:
# List the available record sets and their fields (using @id)
print("Available Record Sets:")
record_sets = []
for record_set in dataset.metadata.record_sets:
    print(f"- RecordSet @id: {record_set.id}, name: {record_set.name}")
    record_sets.append(record_set.id)
    print("  Fields (by @id):")
    for field in record_set.fields:
        print(f"    - Field @id: {field.id}, name: {field.name}, dtype: {field.data_type}")
    print()
if not record_sets:
    print("No record sets found in dataset metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references use the record set and field `@id`s only.

In [ ]:
# If there are no record sets, raise an exception
if not record_sets:
    raise ValueError("No record sets found in Croissant metadata.")
# For demonstration, use the first record set
record_set_id = record_sets[0]

# Load records from the record set using its @id only
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)
print(f"Loaded DataFrame for RecordSet @id: {record_set_id}")
print("Columns (field @ids):", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common filtering, normalization, or grouping steps on a numeric field. All field references are by `@id` only.

We'll select the first numeric field for demonstration.

In [ ]:
# Identify a numeric field (by Croissant @id, type Float or Integer)
numeric_field_id = None
group_field_id = None
first_record_set = None
for record_set in dataset.metadata.record_sets:
    if record_set.id == record_set_id:
        first_record_set = record_set
        for field in record_set.fields:
            if field.data_type in ("Float", "Number", "Integer"):
                numeric_field_id = field.id
                break
        # Try to pick a different field as group key (categorical or string)
        for field in record_set.fields:
            if field.id != numeric_field_id and field.data_type in ("Text", "String"):
                group_field_id = field.id
                break
        break
if numeric_field_id is None:
    raise ValueError("No numeric field found in this record set.")
print(f"Using numeric field (by @id): {numeric_field_id}")
print(f"Using group field (by @id): {group_field_id}")

# Remove outliers: filter only values above a threshold
threshold = df[numeric_field_id].astype(float).mean()
filtered_df = df[df[numeric_field_id].astype(float) > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
fval = filtered_df[numeric_field_id].astype(float)
filtered_df[f"{numeric_field_id}_normalized"] = (fval - fval.mean()) / fval.std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# If suitable, group by another field and calculate statistics
if group_field_id and group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the numeric field and group means if available. All field references by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].astype(float), kde=True, bins=30)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# If grouped, show barplot
if group_field_id and group_field_id in df.columns:
    top_groups = df[group_field_id].value_counts().head(10).index
    plt.figure(figsize=(10,4))
    dftmp = df[df[group_field_id].isin(top_groups)].copy()
    sns.barplot(x=group_field_id, y=numeric_field_id, data=dftmp, estimator=sum)
    plt.title(f"Sum of {numeric_field_id} by {group_field_id} (Top 10)")
    plt.xticks(rotation=90)
    plt.show()

## 6. Conclusion
- This notebook demonstrated a complete workflow leveraging the `mlcroissant` library for programmatic FAIR² dataset access and analysis using Croissant `@id` referencing. 
- Key steps: loading metadata, discovering schema elements, accessing and manipulating records by `@id`, basic EDA, and generating plots for interpretation.

For advanced/specific biological or proteomics workflows, use the identified `@id` fields in downstream analyses.
